# Uczenie reprezentacji wielokanałowego EEG do klasyfikacji choroby Alzheimera

**Raport końcowy - projekt zespołowy (Uczenie Reprezentacji)**

| Osoba | Wkład |
|---|---|
| **Filip** | Potok danych (preprocessing, czyszczenie, cięcie EEG), infrastruktura repozytorium (DVC), baseline (Autoenkoder) |
| **Damian** | Definicja, adaptacja i trening dwóch modeli kontrastywnych (TNC, CPC) + klasyfikacja liniowa |
| **Jakub** | Model przestrzeni stanów Mamba, ewaluacja końcowa, badanie hiperparametrów struktur sieciowych |

---
# 1. Opis problemu

### 1.1. Dane wejściowe
Wielokanałowe sygnały **EEG** ze zbioru **BrainLat**. Surowe nagrania (EEGLAB `.set`) są cięte na **okna 3-sekundowe** i zapisywane jako tensory PyTorch - jeden plik na pacjenta, kształt `(liczba_okien, 128 kanałów, 1536 próbek)`. Okna zachowują **kolejność czasową**, co jest kluczowe dla metod kontrastywnych wykorzystujących strukturę czasu.

### 1.2. Dane wyjściowe
Binarna etykieta diagnozy: **AD** (Alzheimer, `1`) vs **CN** (grupa kontrolna, `0`). Zbiór: **67 pacjentów** (32 CN, 35 AD).

### 1.3. Typ problemu
**Klasyfikacja binarna.** Reprezentacje uczone są **bez etykiet** (samonadzorowanie); etykiety wchodzą dopiero na etapie *klasyfikacji liniowej* (linear evaluation).

### 1.4. Hipoteza badawcza
> Reprezentacje wyuczone metodami kontrastywnymi (TNC, CPC), wykorzystującymi **strukturę czasową** EEG, dadzą cechy lepiej rozdzielające klasy AD/CN niż naiwny baseline (Autoenkoder rekonstrukcyjny) - mierzone wyższym **ROC-AUC** w klasyfikacji liniowej.

### 1.5. Czym jest "klasyfikacja liniowa" (linear evaluation)
Standardowy protokół oceny reprezentacji w uczeniu samonadzorowanym:
1. enkoder uczy się **bez etykiet**;
2. enkoder jest **zamrażany** (wagi nie zmieniają się);
3. na zamrożonych cechach trenujemy **wyłącznie prosty klasyfikator** (liniowy / płytka głowica);
4. mierzymy AUC/accuracy.

Jeśli słaby klasyfikator liniowy potrafi rozdzielić klasy na danych cechach, to znaczy, że **enkoder nauczył się użytecznej, liniowo separowalnej reprezentacji**. Liniowość izoluje wkład samego enkodera.

---
# 2. Dane: wczytanie, preprocessing, przygotowanie
### *(sekcja Filipa - poniżej struktura do uzupełnienia)*

**Do opisania:**
- Źródło i charakterystyka zbioru BrainLat (liczba pacjentów, kanały, częstotliwość próbkowania, długość nagrań).
- Kroki preprocessingu: wczytanie (MNE), filtrowanie częstotliwościowe, usuwanie artefaktów, cięcie na okna 3 s, normalizacja.
- Format wyjściowy (`data/processed/*.pt`), mapowanie etykiet (AD=1, CN=0).
- Podział danych: `StratifiedGroupKFold` na poziomie pacjenta (brak data leakage).

In [ ]:
# (Filip) miejsce na kod preprocessingu / wczytania surowych danych, jeśli ma być w raporcie.
# Przetworzone dane są już dostępne w data/processed/*.pt i wczytywane w sekcji 3.1 poniżej.

---
# 3. Baseline - Autoenkoder
### *(sekcja Filipa - kod i struktura przygotowane, do opisania słownie)*

**Do opisania:**
- **Krótki opis:** konwolucyjny Autoenkoder uczony rekonstrukcją (MSE); enkoder `Conv1d→BN→ReLU→MaxPool` ×2.
- **Odpalenie dla domyślnych wartości** + klasyfikacja liniowa (AUC/acc po foldach) - kod w 3.2.
- **Wyniki i wnioski** - punkt odniesienia (*naiwny baseline*) dla metod kontrastywnych z sekcji 4–5.

Funkcje pomocnicze i wczytanie danych (sekcja 3.1) są współdzielone z częścią Damiana - definiujemy je raz, niżej.

## 3.1. Konfiguracja, wczytanie danych i funkcje wspólne
*(współdzielone przez sekcje 3–5)*

In [ ]:
import os, sys, glob, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, roc_auc_score

ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.models.cpc import CPCModel
from src.models.tnc import TNCModel
from src.models.baseline_ae import Autoencoder
from src.models.linear_classifier import LinearClassifier
from src.data.dataloaders import get_fold_sequence_dataloaders, get_fold_dataloaders
from src.evaluation.encoders import build_encoder
# Wspólna logika treningu SSL i klasyfikacji liniowej - to samo źródło, którego używają
# skrypty CLI src/training/train_cpc.py i train_tnc.py (jedno miejsce prawdy, brak duplikacji).
from src.training.ssl_training import (train_ssl as _train_ssl, linear_probe as _linear_probe,
                                       eval_head as _eval_head)

DATA_DIR = os.path.join(ROOT, 'data', 'processed')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
print('ROOT      :', ROOT)
print('Urządzenie:', DEVICE)
print('Pliki .pt :', len(glob.glob(os.path.join(DATA_DIR, '*.pt'))))

**Przełącznik trybu.** Trening na CPU jest wolny. `QUICK=True` → szybki przebieg do weryfikacji poprawności; `QUICK=False` → pełne wyniki do sprawozdania (najlepiej GPU).

**Zapisywanie / wczytywanie (dla współpracy i DVC).** Artefakty na dysku:
- **wagi enkoderów** → `models/representations/<wariant>/encoder_fold_*.pth` (→ DVC);
- **głowice liniowe** → `models/evaluation/<wariant>/head_{win,pat}_fold_*.pth` (→ DVC) - osobno dla metryki per-okno (`win`) i per-pacjent (`pat`);
- **wyniki liczbowe** (historie, tabele HP/CV, metryki) → `notebooks/results/*.{json,csv}` (→ git).

Wczytywanie steruje się **osobno dla każdego modelu i osobno dla dwóch etapów**:
- `*_LOAD_ENCODER` - wczytaj gotowe **wagi enkodera** zamiast trenować SSL (kontroluje też trening enkoderów w badaniu HP);
- `*_LOAD_PROBE` - nie trenuj głowicy: wczytaj gotowe wyniki (JSON) albo zapisaną **głowicę** i policz metryki samym inference.

Można więc wziąć gotowy enkoder z dysku, ale **przeliczyć ewaluację liniową od nowa** (`*_LOAD_ENCODER=True`, `*_LOAD_PROBE=False`). Gdy plików nie ma, dana część trenuje się od zera niezależnie od flagi.

**Early-stopping.** Każda ewaluacja liniowa (sekcje 4/5 oraz 7.1) przerywa trening głowicy, gdy AUC nie poprawia się przez `PROBE_PATIENCE` epok.

In [ ]:
QUICK = False            # <-- False dla pełnego eksperymentu (najlepiej GPU)

# --- Wczytywanie vs trening: DWIE niezależne decyzje na każdy model -------------------------
# Każdy model ma OSOBNO sterowane:
#   *_LOAD_ENCODER : True  -> wczytaj zapisane wagi enkodera (models/representations/<wariant>/*.pth)
#                             zamiast trenować SSL od zera. Steruje też badaniem HP (siatka/HP study
#                             trenują enkodery, więc tu decyduje flaga enkodera).
#                    False -> zawsze trenuj enkoder (SSL) od zera.
#   *_LOAD_PROBE   : True  -> wczytaj zapisaną ewaluację liniową zamiast trenować głowicę. Źródła w
#                             kolejności: (1) wyniki results/*.json/.csv; (2) zapisane GŁOWICE liniowe
#                             models/evaluation/<wariant>/head_*_fold_*.pth (inference bez treningu).
#                    False -> zawsze dotrenuj głowicę liniową na (wczytanym lub świeżo wytrenowanym)
#                             enkoderze. Kombinacja LOAD_ENCODER=True + LOAD_PROBE=False = "weź gotowy
#                             enkoder, ale przelicz ewaluację liniową od nowa".
# Brak wag/wyników/głowic na dysku => trening od zera niezależnie od flagi (z komunikatem).
AE_LOAD_ENCODER,  AE_LOAD_PROBE  = True, True
TNC_LOAD_ENCODER, TNC_LOAD_PROBE = True, True
CPC_LOAD_ENCODER, CPC_LOAD_PROBE = True, True

if QUICK:
    N_SPLITS, SSL_EPOCHS, PROBE_EPOCHS = 2, 3, 10
    HP_SPLITS, SSL_EPOCHS_FINAL = 2, 3
    PROBE_PATIENCE = 3
else:
    # Wariant BEZ glowicy projekcyjnej (use_projection=False w modelach -- patrz ablacja w 4.1/5.1).
    # 5-fold CV dla wynikow finalnych. Badanie wplywu HP na HP_SPLITS=3 foldach (tansze, a wystarcza
    # do wyboru 2 najlepszych wartosci). Zamiast random search: mala siatka 2x2 z dwoch najlepszych
    # wartosci kazdego z dwoch HP, oceniana na pelnym 5-fold CV. best trenowany dluzej (SSL_EPOCHS_FINAL).
    N_SPLITS, SSL_EPOCHS, PROBE_EPOCHS = 5, 30, 30
    HP_SPLITS, SSL_EPOCHS_FINAL = 3, 40
    PROBE_PATIENCE = 5

# Early-stopping KAZDEJ ewaluacji liniowej (sekcje 4/5 oraz 7.1): trening glowicy przerywamy,
# gdy AUC (per-okno w 4/5, per-pacjent w 7.1) nie poprawi sie przez PROBE_PATIENCE epok.

RESULTS_DIR = os.path.join(ROOT, 'notebooks', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Tryb: {"SZYBKI" if QUICK else "PEŁNY (bez głowicy)"} | n_splits={N_SPLITS} hp_splits={HP_SPLITS} '
      f'ssl={SSL_EPOCHS} ssl_final={SSL_EPOCHS_FINAL} probe={PROBE_EPOCHS} patience={PROBE_PATIENCE}')
print(f'Wczytywanie (enkoder/probe):  AE=({AE_LOAD_ENCODER},{AE_LOAD_PROBE})  '
      f'TNC=({TNC_LOAD_ENCODER},{TNC_LOAD_PROBE})  CPC=({CPC_LOAD_ENCODER},{CPC_LOAD_PROBE})')

In [ ]:
# --- Inspekcja danych: kształt, rozkład klas, brak przecieku pacjentów ---
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.pt')))
labels = [int(os.path.basename(f).split('_label_')[1].replace('.pt','')) for f in files]
n_per_patient = [torch.load(f).shape[0] for f in files]
ex = torch.load(files[0])

print(f'Pacjentów: {len(files)}  (CN={labels.count(0)}, AD={labels.count(1)})')
print(f'Kształt pacjenta: {tuple(ex.shape)}  -> (okna, kanały, próbki)')
print(f'Okien/pacjenta: min={min(n_per_patient)} max={max(n_per_patient)} śr={np.mean(n_per_patient):.0f}')

fold0, tl_seq, vl_seq = next(get_fold_sequence_dataloaders(DATA_DIR, n_splits=N_SPLITS))
leak = set(tl_seq.dataset.subjects) & set(vl_seq.dataset.subjects)
print(f'Fold 0: {len(tl_seq)} train / {len(vl_seq)} val pacjentów | leakage: {len(leak)} (musi być 0)')

fig, ax = plt.subplots(1, 2, figsize=(12, 3.3))
ax[0].hist(n_per_patient, bins=20, color='steelblue', edgecolor='white')
ax[0].set_title('Liczba okien na pacjenta')
ax[0].set_xlabel('okna 3 s')
ax[0].set_ylabel('pacjenci')

for ch in range(4):
    ax[1].plot(ex[0, ch, :300].numpy(), lw=0.8, label=f'kanał {ch}')
ax[1].set_title('Fragment EEG (okno 0, 4 kanały)')
ax[1].set_xlabel('próbka')
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

### Metryki, walidacja i funkcje wspólne

**Metryki:** **ROC-AUC** (wiodąca) i **Accuracy**, średnia ± std po foldach. Dodatkowo per-pacjent: **F1, Precision, Recall** dla klasy AD (sekcja 7.1).

**Walidacja krzyżowa (cała próba, bez osobnego test setu).** `StratifiedGroupKFold` (**5 foldów**) na **poziomie pacjenta**: żaden pacjent nie jest jednocześnie w treningu i walidacji (brak *data leakage*). Wynik = średnia z 5 foldów.
> Przy 67 pacjentach **nie** wydzielamy osobnego zbioru testowego - held-out test miałby ~13 osób (loteria), a tracilibyśmy dane treningowe. CV na całej próbie to standard dla małych zbiorów.

**Dobór hiperparametrów - dwa etapy o różnym koszcie.**
- **Badanie wpływu HP** (4.3 / 5.3) liczymy na **3 foldach** (`HP_SPLITS`) - pokazuje *trend* i wyłania 2 najlepsze wartości każdego parametru.
- **Siatka 2×2** (4.4 / 5.4) - finalny dobór - sprawdza 4 kombinacje (2 najlepsze wartości × 2 najlepsze) na **pełnym 5-fold CV**. Zastępuje random search: deterministyczna, tania, oparta na danych z badania HP.

**Finalny model `best`** trenowany z większą liczbą epok SSL (`SSL_EPOCHS_FINAL=40` vs `SSL_EPOCHS=30`) - finalny wariant zasługuje na dłuższy trening.

**Uwaga o uczciwości.** Siatka 2×2 i wariant „best" korzystają z tego samego CV, na którym raportujemy wynik → lekkie optymistyczne obciążenie „best". Pełna ścisłość wymagałaby *nested CV* (niepraktycznego przy tak małym zbiorze). „best" = orientacyjny sufit, „domyślny" = uczciwy wynik bez strojenia.

**Ablacja głowicy projekcyjnej.** Cały ten przebieg jest **bez głowicy SimCLR** (`use_projection=False`) - wcześniejszy wariant z głowicą obniżał AUC na tym małym zbiorze (patrz 4.1/5.1).

**Funkcje pomocnicze:** `linear_probe()`, `train_ssl()`, `run_cv_smart()` (train-or-load), `eval_n_folds()` (ocena konfiguracji HP), `hp_study()` (3 foldy, trend) + `top2()`, `grid_search()` (siatka 2×2 na pełnym CV).

In [ ]:
# ---------- zapis / odczyt lekkich wyników (-> git) ----------
def _res(name): return os.path.join(RESULTS_DIR, name)
def save_json(o, name):
    with open(_res(name), 'w', encoding='utf-8') as f: json.dump(o, f)
def load_json(name):
    with open(_res(name), 'r', encoding='utf-8') as f: return json.load(f)
def save_df(df, name): df.to_csv(_res(name), index=False)
def load_df(name): return pd.read_csv(_res(name))
def results_exist(*names): return all(os.path.exists(_res(n)) for n in names)

# ---------- zapis / odczyt wag enkoderów (-> DVC) ----------
def _enc_path(variant, fold):
    return os.path.join(ROOT, 'models', 'representations', variant, f'encoder_fold_{fold}.pth')

def save_encoder(model, variant, fold):
    os.makedirs(os.path.dirname(_enc_path(variant, fold)), exist_ok=True)
    torch.save(model.encoder.state_dict(), _enc_path(variant, fold))
    return _enc_path(variant, fold)

def all_encoders_exist(variant, n_splits):
    return all(os.path.exists(_enc_path(variant, f)) for f in range(n_splits))

# ---------- zapis / odczyt GŁOWIC liniowych (-> DVC) ----------
# Głowica = state_dict modułu LinearClassifier.classifier (mała: 128->32->1). Dwa poziomy/fold:
#   level='win' - głowica z epoki max AUC per-OKNO   (zapisywana w sekcjach 4/5, przez run_cv)
#   level='pat' - głowica z epoki max AUC per-PACJENT (zapisywana w sekcji 7.1)
# Mając enkoder (models/representations) + głowicę, metryki liczymy INFERENCE bez treningu.
def _head_path(variant, fold, level):
    return os.path.join(ROOT, 'models', 'evaluation', variant, f'head_{level}_fold_{fold}.pth')

def save_head(head_state, variant, fold, level):
    os.makedirs(os.path.dirname(_head_path(variant, fold, level)), exist_ok=True)
    torch.save(head_state, _head_path(variant, fold, level))
    return _head_path(variant, fold, level)

def load_head(variant, fold, level):
    return torch.load(_head_path(variant, fold, level), map_location=DEVICE)

def all_heads_exist(variant, n_splits, level):
    return all(os.path.exists(_head_path(variant, f, level)) for f in range(n_splits))

# ---------- trening SSL i klasyfikacja liniowa ----------
# Rdzeń (pętla treningu, probe, inference głowicy) jest w src/training/ssl_training.py - to SAMO
# źródło, którego używają skrypty CLI train_cpc.py / train_tnc.py. Tu cienkie wrappery + DEVICE.
# linear_probe: early-stopping (PROBE_PATIENCE) na KAŻDEJ ewaluacji liniowej, zwraca też state_dict
# głowicy z epoki max AUC (do zapisu). eval_head: inference z zapisanej głowicy (bez treningu).
def train_ssl(model, seq_loader, epochs, lr=1e-3, verbose=True):
    return _train_ssl(model, seq_loader, epochs, DEVICE, lr=lr, verbose=verbose)

def linear_probe(encoder, encoded_size, tl, vl, epochs, lr=5e-3, patience=None):
    if patience is None: patience = PROBE_PATIENCE
    return _linear_probe(encoder, encoded_size, tl, vl, epochs, DEVICE, lr=lr, patience=patience)

def eval_head_win(encoder, encoded_size, head_state, vl):
    """Inference per-okno z zapisanej głowicy 'win' (bez treningu). Zwraca (acc, auc)."""
    return _eval_head(encoder, encoded_size, head_state, vl, DEVICE)

# ---------- CV ----------
def run_cv(model_factory, ssl_epochs, n_splits, probe_epochs, lr_ssl=1e-3, verbose=True, save_variant=None):
    accs, aucs, hists = [], [], []
    seq_gen = get_fold_sequence_dataloaders(DATA_DIR, n_splits=n_splits)
    win_gen = get_fold_dataloaders(DATA_DIR, n_splits=n_splits, batch_size=64)
    for (fold, tl_seq, _), (_, tl_win, vl_win) in zip(seq_gen, win_gen):
        s, _ = next(iter(tl_seq))
        C, T = s.shape[1], s.shape[2]
        if verbose: print(f'--- FOLD {fold} (trening) ---')
        m = model_factory(C, T)
        h = train_ssl(m, tl_seq, ssl_epochs, lr=lr_ssl, verbose=verbose)
        if save_variant: save_encoder(m, save_variant, fold)
        m.encoder.eval()
        acc, auc, head = linear_probe(m.encoder, m.encoded_size, tl_win, vl_win, probe_epochs)
        if save_variant: save_head(head, save_variant, fold, 'win')   # głowica per-okno -> models/evaluation
        if verbose: print(f'  -> linear-eval: Acc {acc:.4f} | AUC {auc:.4f}')
        accs.append(acc)
        aucs.append(auc)
        hists.append(h)
    return accs, aucs, hists

def run_cv_pretrained(variant, model_name, n_splits, probe_epochs, load_probe=True, verbose=True):
    """Wczytuje gotowe enkodery (bez treningu SSL). Jeśli load_probe i są zapisane GŁOWICE 'win'
    -> inference bez treningu; inaczej dotrenuj głowicę i zapisz."""
    use_heads = load_probe and all_heads_exist(variant, n_splits, 'win')
    accs, aucs = [], []
    for fold, tl_win, vl_win in get_fold_dataloaders(DATA_DIR, n_splits=n_splits, batch_size=64):
        s, _ = next(iter(tl_win))
        C, T = s.shape[1], s.shape[2]
        enc, es = build_encoder(model_name, C, T, DEVICE)
        enc.load_state_dict(torch.load(_enc_path(variant, fold), map_location=DEVICE))
        enc.eval()
        if use_heads:
            acc, auc = eval_head_win(enc, es, load_head(variant, fold, 'win'), vl_win)
            if verbose: print(f'--- FOLD {fold} (głowica win z dysku -> inference) Acc {acc:.4f} | AUC {auc:.4f}')
        else:
            acc, auc, head = linear_probe(enc, es, tl_win, vl_win, probe_epochs)
            save_head(head, variant, fold, 'win')
            if verbose: print(f'--- FOLD {fold} (enkoder {variant}, probe+zapis głowicy) Acc {acc:.4f} | AUC {auc:.4f}')
        accs.append(acc)
        aucs.append(auc)
    return accs, aucs, []

def run_cv_smart(model_factory, model_name, variant, ssl_epochs, n_splits, probe_epochs,
                 load_encoder, load_probe, lr_ssl=1e-3, hist_file=None, res_file=None):
    """Train-or-load sterowany DWIEMA niezależnymi flagami:
      load_probe   - True i są zapisane wyniki linear-eval -> wczytaj je (bez probe i SSL);
                     w przeciwnym razie probe: jeśli są zapisane GŁOWICE 'win' -> inference, inaczej trening głowicy.
      load_encoder - jeśli probe trzeba przeliczyć: True i są wagi -> wczytaj enkoder (bez SSL),
                     w przeciwnym razie trenuj SSL od zera i zapisz wagi.
    Po przeliczeniu probe zapisujemy wyniki (i historię, jeśli był trening SSL); głowice zapisują run_cv*."""
    # (1) gotowe wyniki linear-eval -> najtańsza ścieżka, probe niepotrzebny
    if load_probe and res_file and results_exist(res_file):
        r = load_json(res_file)
        hist = load_json(hist_file) if (hist_file and results_exist(hist_file)) else []
        print(f'[{variant.upper()}] Wczytano zapisane wyniki linear-eval ({res_file}) - bez probe i treningu.')
        return r['accs'], r['aucs'], hist
    # (2) trzeba przeliczyć probe; enkoder: wczytaj wagi albo trenuj SSL
    if load_encoder and all_encoders_exist(variant, n_splits):
        print(f'[{variant.upper()}] Wczytuję gotowe enkodery (wagi) -> klasyfikacja liniowa (głowice z dysku lub trening).')
        accs, aucs, hist = run_cv_pretrained(variant, model_name, n_splits, probe_epochs, load_probe=load_probe)
    else:
        if load_encoder:
            print(f'[{variant.upper()}] Brak kompletu wag na dysku -> trening SSL od zera.')
        else:
            print(f'[{variant.upper()}] LOAD_ENCODER=False -> trening SSL od zera.')
        accs, aucs, hist = run_cv(model_factory, ssl_epochs, n_splits, probe_epochs, lr_ssl=lr_ssl, save_variant=variant)
    if res_file: save_json({'accs': accs, 'aucs': aucs}, res_file)
    if hist_file and hist: save_json(hist, hist_file)  # historia tylko gdy był trening SSL
    return accs, aucs, hist

# ---------- ocena konfiguracji HP (bez zapisu wag/głowic) ----------
# eval_n_folds: srednie AUC/Acc po `n_folds` foldach dla danej konfiguracji. n_folds=HP_SPLITS dla badania
# wplywu HP (tanio), n_folds=N_SPLITS dla siatki 2x2 (rzetelny wybor finalny na pelnym CV).
# save_variant=None -> HP NIE zapisuje enkoderow ani glowic (to tylko przeglad konfiguracji);
# early-stopping (PROBE_PATIENCE) i tak dziala, bo run_cv wola linear_probe z patience.
def eval_n_folds(model_factory, ssl_epochs, probe_epochs, n_folds, lr_ssl=1e-3):
    accs, aucs, _ = run_cv(model_factory, ssl_epochs, n_folds, probe_epochs,
                           lr_ssl=lr_ssl, verbose=False, save_variant=None)
    return float(np.mean(accs)), float(np.mean(aucs)), float(np.std(aucs))

_METRIC_COLS = ('Accuracy', 'ROC-AUC', 'ROC-AUC_std')

def hp_study(build_model, param_name, grid, fixed, res_file, load_encoder):
    """Badanie WPLYWU jednego hiperparametru na HP_SPLITS foldach (srednia +/- std).

    Pokazuje TREND (jak wartosc parametru wplywa na AUC) i sluzy do wyboru 2 najlepszych wartosci
    do siatki 2x2. HP_SPLITS (3) < N_SPLITS (5) -> tansze niz pelne CV, a wystarcza do rankingu wartosci.
    Badanie trenuje enkodery od zera, więc steruje nim flaga enkodera (load_encoder): True i tabela
    istnieje -> wczytaj zapisaną; inaczej policz."""
    if load_encoder and results_exist(res_file):
        print(f'Wczytano zapisaną tabelę badania HP ({res_file}).'); return load_df(res_file)
    rows = []
    for val in grid:
        print(f'[{param_name} = {val}] ({HP_SPLITS}-fold CV...)')
        params = {**fixed, param_name: val}
        acc, auc, sd = eval_n_folds(lambda C, T, p=params: build_model(C, T, **p), SSL_EPOCHS, PROBE_EPOCHS, HP_SPLITS)
        rows.append({param_name: val, 'Accuracy': round(acc, 4), 'ROC-AUC': round(auc, 4), 'ROC-AUC_std': round(sd, 4)})
        print(f'    -> AUC {auc:.4f} +/- {sd:.4f}')
    df = pd.DataFrame(rows); save_df(df, res_file); return df

def top2(hp_df, param_name):
    """Dwie wartosci parametru o najwyzszym ROC-AUC z tabeli badania HP (do siatki 2x2)."""
    best = hp_df.sort_values('ROC-AUC', ascending=False)[param_name].tolist()
    return best[:2]

def grid_search(build_model, grid_dict, res_file, load_encoder):
    """Pelna siatka kombinacji (tu 2x2 = 4 konfiguracje) oceniana na PELNYM N_SPLITS CV.
    Zastepuje random search: maly, deterministyczny przeglad najlepszych wartosci HP. Trenuje enkodery,
    więc steruje nią flaga enkodera (load_encoder): True i tabela istnieje -> wczytaj; inaczej policz."""
    if load_encoder and results_exist(res_file):
        print(f'Wczytano zapisaną tabelę siatki HP ({res_file}).'); return load_df(res_file)
    import itertools
    keys = list(grid_dict.keys()); rows = []
    for combo in itertools.product(*[grid_dict[k] for k in keys]):
        params = {k: (v.item() if hasattr(v, 'item') else v) for k, v in zip(keys, combo)}
        acc, auc, sd = eval_n_folds(lambda C, T, p=params: build_model(C, T, **p), SSL_EPOCHS, PROBE_EPOCHS, N_SPLITS)
        rows.append({**params, 'Accuracy': round(acc, 4), 'ROC-AUC': round(auc, 4), 'ROC-AUC_std': round(sd, 4)})
        print(f'  {params} -> AUC {auc:.4f} +/- {sd:.4f}')
    df = pd.DataFrame(rows).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
    save_df(df, res_file); return df

def best_params_from(rs_df, int_keys):
    """Wyciąga najlepszy zestaw HP z tabeli siatki (bez kolumn metryk), rzutując wybrane na int."""
    p = {k: rs_df.iloc[0][k] for k in rs_df.columns if k not in _METRIC_COLS}
    for k in int_keys:
        if k in p: p[k] = int(p[k])
    return p

def fmt(vals): return f'{np.mean(vals):.3f} ± {np.std(vals):.3f}'
def plot_hist(hists, title, msg):
    if hists:
        plt.figure(figsize=(6, 3.3))
        for i, h in enumerate(hists): plt.plot(range(1, len(h)+1), h, marker='o', alpha=0.7, label=f'fold {i}')
        plt.title(title); plt.xlabel('epoka'); plt.ylabel('loss'); plt.legend(fontsize=8); plt.tight_layout(); plt.show()
    else:
        print(msg)


## 3.2. Trening baseline'u (AE) + klasyfikacja liniowa

In [ ]:
def train_cv_ae(epochs, n_splits, probe_epochs, lr=1e-3):
    """Trenuje Autoenkoder rekonstrukcją (MSE) na każdym foldzie, zapisuje enkoder, robi
    klasyfikację liniową (early-stopping) i zapisuje głowicę 'win' do models/evaluation.
    Zwraca (accs, aucs). Osobno od run_cv, bo AE uczy się rekonstrukcją, nie zadaniem SSL."""
    accs, aucs = [], []
    for fold, tl, vl in get_fold_dataloaders(DATA_DIR, n_splits=n_splits, batch_size=64):
        s, _ = next(iter(tl))
        C, T = s.shape[1], s.shape[2]
        print(f'--- FOLD {fold} (AE - trening) ---')

        ae = Autoencoder(C, T).to(DEVICE)
        crit = nn.MSELoss()
        opt = optim.Adam(ae.parameters(), lr=lr)

        for ep in range(epochs):
            ae.train()
            for X, _ in tl:
                X = X.to(DEVICE); opt.zero_grad(); _, rec = ae(X)
                if rec.shape[2] != X.shape[2]:
                    rec = nn.functional.interpolate(rec, size=X.shape[2])
                crit(rec, X).backward()
                opt.step()
        save_encoder(ae, 'ae', fold)
        ae.encoder.eval()
        acc, auc, head = linear_probe(ae.encoder, ae.encoded_size, tl, vl, probe_epochs)
        save_head(head, 'ae', fold, 'win')
        print(f'  -> linear-eval: Acc {acc:.4f} | AUC {auc:.4f}')
        accs.append(acc); aucs.append(auc)
    return accs, aucs

def run_cv_ae(epochs, n_splits, probe_epochs, load_encoder, load_probe, lr=1e-3, res_file='ae_default.json'):
    """Train-or-load AE - te same dwie flagi co run_cv_smart:
      load_probe   - są zapisane wyniki linear-eval -> wczytaj (bez probe i treningu); w przeciwnym
                     razie probe: są głowice 'win' -> inference, inaczej trening głowicy + zapis;
      load_encoder - wczytaj wagi enkodera AE (bez rekonstrukcji) zamiast trenować od zera."""
    if load_probe and results_exist(res_file):
        r = load_json(res_file)
        print(f'[AE] Wczytano zapisane wyniki linear-eval ({res_file}) - bez probe i treningu.')
        return r['accs'], r['aucs']
    if load_encoder and all_encoders_exist('ae', n_splits):
        print('[AE] Wczytuję gotowe enkodery (wagi) -> klasyfikacja liniowa (głowice z dysku lub trening).')
        accs, aucs, _ = run_cv_pretrained('ae', 'ae', n_splits, probe_epochs, load_probe=load_probe)
    else:
        if load_encoder:
            print('[AE] Brak kompletu wag na dysku -> trening AE od zera.')
        else:
            print('[AE] AE_LOAD_ENCODER=False -> trening AE od zera.')
        accs, aucs = train_cv_ae(epochs, n_splits, probe_epochs, lr=lr)
    save_json({'accs': accs, 'aucs': aucs}, res_file)
    return accs, aucs

print('=== Baseline AE ===')
ae_acc, ae_auc = run_cv_ae(SSL_EPOCHS, N_SPLITS, PROBE_EPOCHS,
                           load_encoder=AE_LOAD_ENCODER, load_probe=AE_LOAD_PROBE)
print(f'\nAE | Acc {fmt(ae_acc)} | AUC {fmt(ae_auc)}')

---
# 4. TNC - Temporal Neighborhood Coding

## 4.1. Krótki opis
- **Publikacja:** Tonekaboni, Eytan, Goldenberg, *Unsupervised Representation Learning for Time Series with Temporal Neighborhood Coding*, ICLR 2021 (arXiv:2106.00750).
- **Idea:** dla okna kotwicznego `t` okna z **sąsiedztwa czasowego** są pozytywne, a odległe - negatywne. Dyskryminator uczy się je rozróżniać, co wymusza spójność reprezentacji w czasie. Oryginał stosuje **PU-learning**: odległe okna są *nieoznaczone* → strata negatywna ważona współczynnikiem `w`.
- **Nasze adaptacje / wprowadzone zmiany** (względem oryginału):
  1. enkoder splotowy 1D **wspólny dla TNC i CPC** (`src/models/encoder.py` - 4 bloki Conv1d 32→64→128→128 + GELU + global average pooling → reprezentacja **128-d**). Ten sam enkoder w obu metodach sprawia, że porównanie TNC vs CPC ocenia **samą metodę SSL**, a nie różnice architektury. (Oryginalne lab. TNC używa dwukierunkowego GRU - my adaptujemy do EEG, gdzie okno to wielokanałowy sygnał 128×1536, nie krótka sekwencja);
  2. sąsiedztwo = prosty promień `neighbor_range` (uproszczenie wobec gaussowskiego `t*~N(t, η·δ)` + testu ADF z oryginału);
  3. **ablacja głowicy projekcyjnej (SimCLR).** Sprawdziliśmy wariant z głowicą MLP 128→128→64 (strata na `g(z)`, ewaluacja na surowym 128-d). Na tym **małym zbiorze (53 pacjentów treningowych) głowica systematycznie obniżała AUC** - efekt znany: SimCLR pomaga przy dużych zbiorach z mocną augmentacją, nie w tym reżimie. **Finalnie głowicy nie używamy** (`use_projection=False`).
  
  Implementacja modelu: `src/models/tnc.py`.

**Hiperparametry:** `neighbor_range`, `num_samples`, `w`.

## 4.2. Odpalenie dla wartości domyślnych

In [ ]:
print('=== TNC (domyślne: neighbor_range=3, num_samples=5, w=0.05) ===')
t0 = time.time()
tnc_acc, tnc_auc, tnc_hist = run_cv_smart(
    lambda C, T: TNCModel(C, T, neighbor_range=3, num_samples=5, w=0.05),
    model_name='tnc', variant='tnc', ssl_epochs=SSL_EPOCHS, n_splits=N_SPLITS, probe_epochs=PROBE_EPOCHS,
    load_encoder=TNC_LOAD_ENCODER, load_probe=TNC_LOAD_PROBE,
    hist_file='tnc_default_hist.json', res_file='tnc_default.json',
)
print(f'\nTNC (domyślny) | Acc {fmt(tnc_acc)} | AUC {fmt(tnc_auc)}  ({time.time()-t0:.0f}s)')

In [ ]:
plot_hist(tnc_hist, 'TNC - strata dyskryminatora',
          'Wczytano gotowe wyniki - brak historii treningu (ustaw TNC_LOAD_ENCODER=False, by ją zobaczyć).')

## 4.3. Badanie hiperparametrów

Dwa najważniejsze, specyficzne dla TNC hiperparametry badamy osobno (analogicznie do oryginalnego laboratorium TNC, gdzie badano `η`/sąsiedztwo oraz `w`). **Badanie wpływu liczymy na 3 foldach** (`HP_SPLITS`) - pokazuje *trend* i wyłania 2 najlepsze wartości każdego parametru do siatki 2×2 (sekcja 4.4).

> **Uwaga o `window_size`.** W oryginalnym laboratorium badano też `window_size`, ale u nas długość okna (3 s) jest **ustalona w preprocessingu** (`make_dataset`, sekcja Filipa) - okna są wycinane raz z surowego EEG, zanim model je zobaczy. Zmiana wymagałaby regeneracji całego zbioru, więc nie jest hiperparametrem modelu TNC i tu jej nie strojimy.

### 4.3a. Wpływ `neighbor_range`
Promień sąsiedztwa decyduje, jak "blisko" w czasie okna uznajemy za podobne (odpowiednik `η` z publikacji). Wynik zapisywany do `results/tnc_hp.csv`.

In [ ]:
tnc_hp_df = hp_study(lambda C, T, **p: TNCModel(C, T, **p),
                     'neighbor_range', [1, 3, 5, 10], {'num_samples': 5, 'w': 0.05}, 'tnc_hp.csv',
                     load_encoder=TNC_LOAD_ENCODER)
tnc_hp_df

In [ ]:
plt.figure(figsize=(6, 3.3))
plt.errorbar(tnc_hp_df['neighbor_range'], tnc_hp_df['ROC-AUC'], yerr=tnc_hp_df['ROC-AUC_std'],
             marker='o', color='indianred', capsize=4, label='ROC-AUC (3-fold)')
plt.plot(tnc_hp_df['neighbor_range'], tnc_hp_df['Accuracy'], marker='s', color='gray', label='Accuracy (3-fold)')
plt.axhline(0.5, ls='--', c='k', lw=1)
plt.title('TNC - wpływ neighbor_range (trend, 3-fold)'); plt.xlabel('neighbor_range'); plt.ylabel('metryka (3-fold CV)')
plt.legend(); plt.tight_layout(); plt.show()

### 4.3b. Wpływ `w` (waga PU-learning)

Współczynnik `w` (Positive-Unlabeled learning) określa, w jakim stopniu okna **spoza** sąsiedztwa traktujemy jako potencjalnie pozytywne: strata negatywna to `w·BCE(·,1) + (1−w)·BCE(·,0)`. `w=0` → odległe okna są czysto negatywne; `w→1` → prawie pozytywne (model traci sygnał kontrastywny). Badamy wpływ na ROC-AUC (1 fold, trend). Wynik w `results/tnc_hp_w.csv`.

In [ ]:
tnc_hp_w_df = hp_study(lambda C, T, **p: TNCModel(C, T, **p),
                       'w', [0.0, 0.05, 0.1, 0.2], {'neighbor_range': 3, 'num_samples': 5}, 'tnc_hp_w.csv',
                       load_encoder=TNC_LOAD_ENCODER)
tnc_hp_w_df

In [ ]:
plt.figure(figsize=(6, 3.3))
plt.errorbar(tnc_hp_w_df['w'], tnc_hp_w_df['ROC-AUC'], yerr=tnc_hp_w_df['ROC-AUC_std'],
             marker='o', color='indianred', capsize=4, label='ROC-AUC (3-fold)')
plt.plot(tnc_hp_w_df['w'], tnc_hp_w_df['Accuracy'], marker='s', color='gray', label='Accuracy (3-fold)')
plt.axhline(0.5, ls='--', c='k', lw=1)
plt.title('TNC - wpływ w (waga PU-learning) (trend, 3-fold)'); plt.xlabel('w'); plt.ylabel('metryka (3-fold CV)')
plt.legend(); plt.tight_layout(); plt.show()

## 4.4. Dobór hiperparametrów - siatka 2×2

Zamiast kosztownego random search bierzemy **2 najlepsze wartości** `neighbor_range` i **2 najlepsze** `w` z badania wpływu (4.3) i sprawdzamy wszystkie **4 kombinacje** na **pełnym 5-fold CV**. Najlepszy zestaw → wariant `tnc_best`. To deterministyczne, tanie (4 konfiguracje) i opiera wybór na danych z badania HP. Wynik w `results/tnc_rs.csv`.

In [ ]:
# 2 najlepsze wartosci kazdego HP z badania wplywu (4.3a neighbor_range, 4.3b w) -> siatka 2x2
tnc_grid = {'neighbor_range': top2(tnc_hp_df, 'neighbor_range'),
            'w':             top2(tnc_hp_w_df, 'w'),
            'num_samples':   [5]}   # num_samples ustalone (mniej wplywowe); siatka 2x2 po nr x w
print('=== TNC siatka 2x2 ===  neighbor_range:', tnc_grid['neighbor_range'], '| w:', tnc_grid['w'])
tnc_rs = grid_search(lambda C, T, **p: TNCModel(C, T, **p), tnc_grid, 'tnc_rs.csv',
                     load_encoder=TNC_LOAD_ENCODER)
tnc_rs

### TNC z najlepszymi hiperparametrami - pełne CV
Najlepszy zestaw z siatki 2×2 dotrenowujemy na **pełnym 5-fold CV**, z **większą liczbą epok SSL** (`SSL_EPOCHS_FINAL`) - finalny model zasługuje na dłuższy trening. Wariant `tnc_best` trafia do tabeli porównawczej (sekcja 7).

In [ ]:
tnc_best_params = best_params_from(tnc_rs, int_keys=('neighbor_range', 'num_samples'))
print('Najlepsze HP TNC:', tnc_best_params, f'| epoki SSL = {SSL_EPOCHS_FINAL}')
t0 = time.time()
tnc_best_acc, tnc_best_auc, tnc_best_hist = run_cv_smart(
    lambda C, T: TNCModel(C, T, **tnc_best_params),
    model_name='tnc', variant='tnc_best', ssl_epochs=SSL_EPOCHS_FINAL, n_splits=N_SPLITS, probe_epochs=PROBE_EPOCHS,
    load_encoder=TNC_LOAD_ENCODER, load_probe=TNC_LOAD_PROBE,
    hist_file='tnc_best_hist.json', res_file='tnc_best.json',
)
print(f'\nTNC (najlepszy) | Acc {fmt(tnc_best_acc)} | AUC {fmt(tnc_best_auc)}  ({time.time()-t0:.0f}s)')

## 4.5. Wyniki i wnioski - TNC

**Domyślny TNC vs baseline AE.** Domyślny TNC (`neighbor_range=3, w=0.05`) osiąga **per-okno AUC 0.599 ± 0.146** wobec **0.580 ± 0.111** dla autoenkodera. Przewaga jest więc niewielka i mieści się w przedziale odchyleń — w wariancie *domyślnym* sama strata sąsiedztwa czasowego nie daje jeszcze wyraźnego zysku. Dopiero strojenie hiperparametrów ujawnia potencjał metody (niżej).

**Wpływ `neighbor_range`.** W badaniu wpływu (3-fold) najlepsza była wartość **3** (AUC 0.683 ± 0.029), z wyraźnym spadkiem dla `nr=5` (0.541) i niestabilnym `nr=10` (0.607 ± 0.144). Jednak na pełnym **5-fold CV w siatce 2×2** najlepszy okazał się **`nr=10`** (AUC 0.719 ± 0.116). Ta rozbieżność rankingu 3-fold vs 5-fold to efekt małego zbioru i dużej wariancji między foldami (wysokie std) - przy tak małej próbie 3 foldy dają tylko zgrubny trend. Interpretacja merytoryczna: zbyt wąskie sąsiedztwo (`nr=1`, AUC 0.585) traktuje jako pozytywne tylko bezpośrednio sąsiadujące okna i daje słaby sygnał kontrastywny, a szersze sąsiedztwo (`nr=10`) lepiej oddaje wolnozmienną strukturę EEG.

**Wpływ `w` (waga PU-learning).** Najlepsza wartość to **`w=0.05`** (AUC 0.557); skrajne `w=0.0` (0.499) i `w=0.10` (0.478) spadają do poziomu losowego. Potwierdza to rolę PU-learningu z oryginału: lekkie traktowanie odległych okien jako *nieoznaczonych* (a nie twardo negatywnych) pomaga, ale zbyt duża waga niszczy sygnał kontrastywny.

**Zysk z optymalizacji.** Najlepszy zestaw (`neighbor_range=10, w=0.05`, SSL 40 epok) podnosi per-okno AUC do **0.676 ± 0.121** (z 0.599 domyślnego, **+0.077**) i jest wyraźnie powyżej baseline'u AE. Na poziomie **per-pacjenta** (sekcja 7.1, metryka rozstrzygająca) zysk jest jeszcze wyraźniejszy: **0.529 → 0.630 AUC**. TNC `best` to najlepsza reprezentacja w całym porównaniu (sekcja 7).

> *Uwaga metodologiczna:* siatka 2×2 i wariant `best` korzystają z tego samego CV, na którym raportujemy wynik (brak nested CV - niepraktyczne przy 67 pacjentach), więc `best` jest lekko optymistycznie obciążony. Wysokie std (±0.12-0.18) wynika z małej liczby pacjentów na fold - różnice rzędu 0.05-0.08 AUC traktujemy jako orientacyjne, nie istotne statystycznie.

---
# 5. CPC - Contrastive Predictive Coding

## 5.1. Krótki opis
- **Publikacja:** van den Oord, Li, Vinyals, *Representation Learning with Contrastive Predictive Coding*, arXiv:1807.03748 (2018).
- **Idea:** enkoder `g_enc` koduje okno do `z_t`; autoregresor `g_ar` (**GRU**) buduje kontekst `c_t`. Dla horyzontu `k` projekcja `W_k` przewiduje `z_{t+k}`. Trening stratą **InfoNCE**.
- **Nasze adaptacje / wprowadzone zmiany:**
  1. enkoder splotowy **wspólny z TNC** (`src/models/encoder.py`, 128-d, global pooling) - patrz uwaga w 4.1;
  2. negatywy losowane wewnątrz sekwencji jednego pacjenta (in-batch negatives);
  3. **ablacja głowicy projekcyjnej (SimCLR)** - jak w TNC (4.1): sprawdzona, na tym zbiorze pogarsza wynik, więc finalnie `use_projection=False`.
  
  Implementacja modelu: `src/models/cpc.py`.

**Hiperparametry:** `prediction_steps`, `context_dim`.

## 5.2. Odpalenie dla wartości domyślnych

In [ ]:
print('=== CPC (domyślne: context_dim=128, prediction_steps=4) ===')
t0 = time.time()
cpc_acc, cpc_auc, cpc_hist = run_cv_smart(
    lambda C, T: CPCModel(C, T, context_dim=128, prediction_steps=4),
    model_name='cpc', variant='cpc', ssl_epochs=SSL_EPOCHS, n_splits=N_SPLITS, probe_epochs=PROBE_EPOCHS,
    load_encoder=CPC_LOAD_ENCODER, load_probe=CPC_LOAD_PROBE,
    hist_file='cpc_default_hist.json', res_file='cpc_default.json',
)
print(f'\nCPC (domyślny) | Acc {fmt(cpc_acc)} | AUC {fmt(cpc_auc)}  ({time.time()-t0:.0f}s)')

In [ ]:
plot_hist(cpc_hist, 'CPC - strata InfoNCE',
          'Wczytano gotowe wyniki - brak historii treningu (ustaw CPC_LOAD_ENCODER=False, by ją zobaczyć).')

## 5.3. Badanie hiperparametrów

Dwa hiperparametry CPC badamy osobno na **3 foldach** (`HP_SPLITS`) - pokazuje trend i wyłania 2 najlepsze wartości każdego do siatki 2×2 (5.4).

### 5.3a. Wpływ `prediction_steps`
Liczba kroków predykcji w przód. Większa wartość = dłuższe zależności czasowe, ale trudniejsze zadanie. Wynik w `results/cpc_hp.csv`.

In [ ]:
cpc_hp_df = hp_study(lambda C, T, **p: CPCModel(C, T, **p),
                     'prediction_steps', [1, 2, 4, 8], {'context_dim': 128}, 'cpc_hp.csv',
                     load_encoder=CPC_LOAD_ENCODER)
cpc_hp_df

In [ ]:
plt.figure(figsize=(6, 3.3))
plt.plot(cpc_hp_df['prediction_steps'], cpc_hp_df['ROC-AUC'], marker='o', color='steelblue', label='ROC-AUC (3-fold)')
plt.plot(cpc_hp_df['prediction_steps'], cpc_hp_df['Accuracy'], marker='s', color='gray', label='Accuracy (3-fold)')
plt.axhline(0.5, ls='--', c='k', lw=1)
plt.title('CPC - wpływ prediction_steps (trend, 3-fold)'); plt.xlabel('prediction_steps'); plt.ylabel('metryka (3-fold CV)')
plt.legend(); plt.tight_layout(); plt.show()

### 5.3b. Wpływ `context_dim`

Wymiar kontekstu autoregresora GRU (`c_t`). Większy = pojemniejszy kontekst, ale więcej parametrów i ryzyko przeuczenia na małym zbiorze. Badamy wpływ na ROC-AUC (3 foldy, trend). Wynik w `results/cpc_hp_ctx.csv`.

In [ ]:
cpc_hp_ctx_df = hp_study(lambda C, T, **p: CPCModel(C, T, **p),
                         'context_dim', [32, 64, 128, 256], {'prediction_steps': 4}, 'cpc_hp_ctx.csv',
                         load_encoder=CPC_LOAD_ENCODER)
cpc_hp_ctx_df

In [ ]:
plt.figure(figsize=(6, 3.3))
plt.plot(cpc_hp_ctx_df['context_dim'], cpc_hp_ctx_df['ROC-AUC'], marker='o', color='steelblue', label='ROC-AUC (3-fold)')
plt.plot(cpc_hp_ctx_df['context_dim'], cpc_hp_ctx_df['Accuracy'], marker='s', color='gray', label='Accuracy (3-fold)')
plt.axhline(0.5, ls='--', c='k', lw=1)
plt.title('CPC - wpływ context_dim (trend, 3-fold)'); plt.xlabel('context_dim'); plt.ylabel('metryka (3-fold CV)')
plt.legend(); plt.tight_layout(); plt.show()

## 5.4. Dobór hiperparametrów - siatka 2×2

2 najlepsze `prediction_steps` (5.3a) × 2 najlepsze `context_dim` (5.3b), wszystkie **4 kombinacje** na **pełnym 5-fold CV**. Najlepszy → `cpc_best`. Wynik w `results/cpc_rs.csv`.

In [ ]:
# 2 najlepsze wartosci kazdego HP z badania wplywu (5.3a prediction_steps, 5.3b context_dim) -> siatka 2x2
cpc_grid = {'prediction_steps': top2(cpc_hp_df, 'prediction_steps'),
            'context_dim':      top2(cpc_hp_ctx_df, 'context_dim')}
print('=== CPC siatka 2x2 ===  prediction_steps:', cpc_grid['prediction_steps'], '| context_dim:', cpc_grid['context_dim'])
cpc_rs = grid_search(lambda C, T, **p: CPCModel(C, T, **p), cpc_grid, 'cpc_rs.csv',
                     load_encoder=CPC_LOAD_ENCODER)
cpc_rs

### CPC z najlepszymi hiperparametrami - pełne CV
Najlepszy zestaw z siatki 2×2 dotrenowany na **pełnym 5-fold CV**, z większą liczbą epok SSL (`SSL_EPOCHS_FINAL`). Wariant `cpc_best`.

In [ ]:
cpc_best_params = best_params_from(cpc_rs, int_keys=('prediction_steps', 'context_dim'))
print('Najlepsze HP CPC:', cpc_best_params, f'| epoki SSL = {SSL_EPOCHS_FINAL}')
t0 = time.time()
cpc_best_acc, cpc_best_auc, cpc_best_hist = run_cv_smart(
    lambda C, T: CPCModel(C, T, **cpc_best_params),
    model_name='cpc', variant='cpc_best', ssl_epochs=SSL_EPOCHS_FINAL, n_splits=N_SPLITS, probe_epochs=PROBE_EPOCHS,
    load_encoder=CPC_LOAD_ENCODER, load_probe=CPC_LOAD_PROBE,
    hist_file='cpc_best_hist.json', res_file='cpc_best.json',
)
print(f'\nCPC (najlepszy) | Acc {fmt(cpc_best_acc)} | AUC {fmt(cpc_best_auc)}  ({time.time()-t0:.0f}s)')

## 5.5. Wyniki i wnioski - CPC

**Domyślny CPC vs baseline AE.** Domyślny CPC (`context_dim=128, prediction_steps=4`) osiąga **per-okno AUC 0.632 ± 0.145** wobec **0.580 ± 0.111** dla autoenkodera - przewaga większa i bardziej konsekwentna niż u domyślnego TNC. Predykcja przyszłych okien (InfoNCE) wymusza cechy niosące informację o dynamice sygnału, użyteczniejsze niż czysta rekonstrukcja AE.

**Wpływ `prediction_steps` (horyzont predykcji).** Zależność jest **niemonotoniczna**: najlepsze są skrajne wartości - **`steps=1`** (AUC 0.658 ± 0.053) i `steps=8` (0.657), a środek (`2`, `4`) wypada słabiej (~0.59). Krótki horyzont (`steps=1`) daje najłatwiejsze, najstabilniejsze zadanie pretekstowe, które na małym zbiorze uczy się najpewniej; bardzo długi (`8`) wymusza cechy długozasięgowe, ale kosztem wariancji. Siatka 2×2 wskazała `steps=1` jako finał.

**Wpływ `context_dim`.** Optymalny jest **`context_dim=128`** (AUC 0.660 ± 0.050); zbyt mały kontekst (`64` → 0.534) gubi informację, a `256` (0.610) nie poprawia wyniku i grozi przeuczeniem GRU na małej próbie. 128 to dobry kompromis pojemność/regularyzacja.

**Zysk z optymalizacji.** Najlepszy zestaw (`prediction_steps=1, context_dim=128`, SSL 40 epok) daje per-okno AUC **0.662 ± 0.107** (z 0.632 domyślnego, **+0.030**) - zysk mniejszy niż u TNC, bo CPC już domyślnie był blisko optimum. Na poziomie **per-pacjenta**: **0.549 → 0.582 AUC**. CPC plasuje się **między** TNC a baseline'em (sekcja 7): lepszy od AE, ale słabszy od strojonego TNC.

> *Uwaga metodologiczna:* jak w 4.5 - `best` dzieli CV z raportowaniem (lekkie obciążenie optymistyczne), a wysokie std przy 67 pacjentach każe traktować różnice rzędu 0.03-0.05 AUC jako orientacyjne.

---
# 6. Mamba - model przestrzeni stanów

---
# 7. Podsumowanie zbiorcze

Tabela i wykres zestawiające metody w wariancie **domyślnym i najlepszym** (po optymalizacji hiperparametrów). Wymagana przez specyfikację tabela głównego eksperymentu z porównaniem do naiwnego baseline'u.

In [ ]:
summary = pd.DataFrame([
    {'Model': 'Baseline (AE)', 'Acc (domyślny)': fmt(ae_acc), 'AUC (domyślny)': fmt(ae_auc),
     'Acc (najlepszy)': '-', 'AUC (najlepszy)': '-', '_auc': np.mean(ae_auc)},
    {'Model': 'TNC', 'Acc (domyślny)': fmt(tnc_acc), 'AUC (domyślny)': fmt(tnc_auc),
     'Acc (najlepszy)': fmt(tnc_best_acc), 'AUC (najlepszy)': fmt(tnc_best_auc), '_auc': np.mean(tnc_best_auc)},
    {'Model': 'CPC', 'Acc (domyślny)': fmt(cpc_acc), 'AUC (domyślny)': fmt(cpc_auc),
     'Acc (najlepszy)': fmt(cpc_best_acc), 'AUC (najlepszy)': fmt(cpc_best_auc), '_auc': np.mean(cpc_best_auc)},
]).sort_values('_auc', ascending=False).drop(columns='_auc').reset_index(drop=True)
print('Tabela główna - klasyfikacja liniowa, AUC/Acc (średnia ± std po CV):')
print('("najlepszy" = po optymalizacji hiperparametrów random searchem)')
summary

In [ ]:
names    = ['Baseline (AE)', 'TNC', 'CPC']
auc_def  = [np.mean(ae_auc), np.mean(tnc_auc), np.mean(cpc_auc)]
std_def  = [np.std(ae_auc),  np.std(tnc_auc),  np.std(cpc_auc)]
auc_best = [np.mean(ae_auc), np.mean(tnc_best_auc), np.mean(cpc_best_auc)]
std_best = [np.std(ae_auc),  np.std(tnc_best_auc),  np.std(cpc_best_auc)]
x = np.arange(len(names)); w = 0.38
plt.figure(figsize=(7, 4))
plt.bar(x - w/2, auc_def,  w, yerr=std_def,  capsize=5, label='domyślny', color='lightsteelblue', edgecolor='white')
plt.bar(x + w/2, auc_best, w, yerr=std_best, capsize=5, label='najlepszy (po opt. HP)', color='steelblue', edgecolor='white')
plt.axhline(0.5, ls='--', c='k', lw=1, label='losowy (AUC=0.5)')
plt.xticks(x, names); plt.ylabel('ROC-AUC (CV)'); plt.ylim(0, 1)
plt.title('Porównanie reprezentacji - klasyfikacja liniowa'); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 7.1. Ewaluacja per-pacjent (agregacja okien)

Dotąd AUC liczyliśmy **per okno**. Ale diagnoza dotyczy **pacjenta**, a jeden pacjent ma 47–225 okien (które **nie są** niezależnymi próbkami). Bardziej poprawna metodologicznie i klinicznie ocena: agregujemy predykcje okien danego pacjenta (**średnie prawdopodobieństwo**) do jednej predykcji na pacjenta, a AUC liczymy na poziomie pacjentów.

Prezentujemy **dwie tabele** z różnych źródeł:
- **per-okno** - bierzemy **gotowe wyniki z sekcji 4/5** (`*_default.json` / `*_best.json`): te same liczby co linear-eval (max AUC po epokach). Tu nic nie trenujemy ponownie.
- **per-pacjent** - **osobna głowica liniowa** na zamrożonym enkoderze (sterowane flagą `*_LOAD_PROBE`). Metryki z **najlepszej epoki wg AUC per-pacjent** + **early-stopping** (`PROBE_PATIENCE`). 

**Zapisywanie głowic.** Każda ewaluacja liniowa zapisuje wytrenowaną głowicę (state_dict warstwy klasyfikatora) do `models/evaluation/<wariant>/`:
- `head_win_fold_*.pth` - głowica z epoki max AUC per-okno (sekcje 4/5);
- `head_pat_fold_*.pth` - głowica z epoki max AUC per-pacjent (ta sekcja).

Dzięki temu kolejny przebieg z `*_LOAD_PROBE=True` **nie trenuje** głowicy - wczytuje ją z dysku i liczy metryki samym **inference** (sekundy). Kolejność źródeł: (1) gotowe liczby w JSON; (2) zapisana głowica + inference; (3) trening od zera. (Wagi głowic, jak enkodery, idą do DVC.)

In [ ]:
import copy
from sklearn.metrics import f1_score, precision_score, recall_score

# Flaga "wczytaj zapisany probe" per wariant: per-patient reużywa enkodera, więc decyduje
# flaga PROBE danego modelu.
_PP_LOAD_PROBE = {'ae': AE_LOAD_PROBE,
                  'tnc': TNC_LOAD_PROBE, 'tnc_best': TNC_LOAD_PROBE,
                  'cpc': CPC_LOAD_PROBE, 'cpc_best': CPC_LOAD_PROBE}
# Per-OKNO bierzemy GOTOWE z wynikow sekcji 4/5 (te same liczby co tam: best-AUC po epokach,
# zapisane jako aucs/accs). NIE trenujemy tu glowicy dla okna - tylko dla pacjenta (nizej).
_PERWIN_FILE = {'ae': 'ae_default.json',
                'tnc': 'tnc_default.json', 'tnc_best': 'tnc_best.json',
                'cpc': 'cpc_default.json', 'cpc_best': 'cpc_best.json'}
# komplet kluczy cache per-PACJENT (osobna zmienna wynikowa, tylko poziom pacjenta).
_PP_KEYS = [f'pat_{m}' for m in ('auc', 'acc', 'f1', 'prec', 'rec')]


def _patient_metrics(clf, vl_seq):
    """Inference per-pacjent: srednie prawdopodobienstwo okien pacjenta -> 1 predykcja/pacjent.
    Zwraca dict metryk (AUC/Acc/F1/Prec/Recall, AD=1). clf musi byc w trybie eval przed wywolaniem."""
    pp, pl = [], []
    with torch.no_grad():
        for seq, label in vl_seq:
            pr = torch.sigmoid(clf(seq.to(DEVICE))).cpu().numpy().ravel()
            pp.append(float(pr.mean())); pl.append(int(label))
    pred = [int(x > 0.5) for x in pp]
    return {'auc': roc_auc_score(pl, pp), 'acc': accuracy_score(pl, pred),
            'f1': f1_score(pl, pred, zero_division=0), 'prec': precision_score(pl, pred, zero_division=0),
            'rec': recall_score(pl, pred, zero_division=0)}


def eval_per_patient(variant, model_name, load_probe, probe_epochs=PROBE_EPOCHS,
                     patience=None, res_file=None):
    """Metryki PER-PACJENT dla zapisanego enkodera wariantu. Trzy ścieżki (wg load_probe):
      (1) jest kompletny cache JSON (res_file) -> wczytaj liczby, bez inference i treningu;
      (2) są zapisane GŁOWICE 'pat' (models/evaluation) -> inference z głowicy, bez treningu;
      (3) inaczej -> dotrenuj głowicę liniową (early-stopping na pat_auc), zapisz głowicę 'pat'.

    BEST PO EPOKACH wg pat_auc + EARLY STOPPING (patience=PROBE_PATIENCE): z epoki o najlepszym
    pat_auc bierzemy KOMPLET metryk pacjenta ORAZ state_dict głowicy (do zapisu/odtworzenia bez treningu).
    Per-okno NIE liczymy tutaj - bierzemy gotowe z sekcji 4/5 (patrz _PERWIN_FILE w petli ponizej)."""
    if patience is None: patience = PROBE_PATIENCE
    # (1) gotowy cache liczb
    if load_probe and res_file and results_exist(res_file):
        cached = load_json(res_file)
        if all(k in cached for k in _PP_KEYS):
            print(f'[{variant}] Wczytano zapisany per-patient ({res_file}).'); return cached
        print(f'[{variant}] Stary/niekompletny cache ({res_file}) -> przeliczam.')
    use_heads = load_probe and all_heads_exist(variant, N_SPLITS, 'pat')   # (2) inference z głowic
    pat = {'auc': [], 'acc': [], 'f1': [], 'prec': [], 'rec': []}
    seq_gen = get_fold_sequence_dataloaders(DATA_DIR, n_splits=N_SPLITS)
    win_gen = get_fold_dataloaders(DATA_DIR, n_splits=N_SPLITS, batch_size=64)
    for (fold, _, vl_seq), (_, tl_win, _) in zip(seq_gen, win_gen):
        s, _ = next(iter(tl_win)); C, T = s.shape[1], s.shape[2]
        enc, es = build_encoder(model_name, C, T, DEVICE)
        enc.load_state_dict(torch.load(_enc_path(variant, fold), map_location=DEVICE)); enc.eval()
        clf = LinearClassifier(enc, es).to(DEVICE)   # enkoder zamrozony w LinearClassifier
        if use_heads:
            # (2) wczytaj głowicę 'pat' i policz metryki inference (bez treningu)
            clf.classifier.load_state_dict(load_head(variant, fold, 'pat')); clf.eval()
            best_p = _patient_metrics(clf, vl_seq)
            print(f'[{variant}] FOLD {fold}: głowica pat z dysku -> inference (AUC {best_p["auc"]:.4f})')
        else:
            # (3) trening głowicy z early-stoppingiem na pat_auc; zapamietaj best-epoka state_dict
            crit = nn.BCEWithLogitsLoss(); opt = optim.Adam(clf.classifier.parameters(), lr=5e-3)
            best_p = {'auc': -1.0}; best_head = copy.deepcopy(clf.classifier.state_dict()); no_improve = 0
            for _ in range(probe_epochs):
                clf.train()
                for X, y in tl_win:
                    X = X.to(DEVICE); y = y.float().to(DEVICE).unsqueeze(1)
                    opt.zero_grad(); crit(clf(X), y).backward(); opt.step()
                clf.eval()
                p = _patient_metrics(clf, vl_seq)
                if p['auc'] > best_p['auc']:
                    best_p = p; best_head = copy.deepcopy(clf.classifier.state_dict()); no_improve = 0
                else:
                    no_improve += 1
                    if no_improve >= patience:   # brak poprawy pat_auc przez `patience` epok -> stop
                        break
            save_head(best_head, variant, fold, 'pat')   # głowica per-pacjent -> models/evaluation
            print(f'[{variant}] FOLD {fold}: trening+zapis głowicy pat (AUC {best_p["auc"]:.4f})')
        for k in pat: pat[k].append(best_p[k])
    out = {f'pat_{k}': v for k, v in pat.items()}
    if res_file: save_json(out, res_file)
    return out

_PP_VARIANTS = [('Baseline (AE)', 'ae', 'ae'),
                ('TNC',          'tnc', 'tnc'),
                ('TNC best',     'tnc_best', 'tnc'),
                ('CPC',          'cpc', 'cpc'),
                ('CPC best',     'cpc_best', 'cpc')]

win_rows, pat_rows = [], []
for name, variant, model in _PP_VARIANTS:
    print(f'=== per-patient: {name} ===')
    # per-OKNO: gotowe best-AUC/Acc z sekcji 4/5 (bez treningu glowicy tutaj)
    w = load_json(_PERWIN_FILE[variant])
    win_rows.append({'Model': name, 'AUC': fmt(w['aucs']), 'Acc': fmt(w['accs']), '_auc': np.mean(w['aucs'])})
    # per-PACJENT: cache -> glowica 'pat' z dysku -> trening glowicy (early-stopping); cache perpat_*.json
    r = eval_per_patient(variant, model, load_probe=_PP_LOAD_PROBE[variant], res_file=f'perpat_{variant}.json')
    pat_rows.append({'Model': name, 'AUC': fmt(r['pat_auc']), 'Acc': fmt(r['pat_acc']),
                     'F1 (AD)': fmt(r['pat_f1']), 'Precision (AD)': fmt(r['pat_prec']),
                     'Recall/czułość (AD)': fmt(r['pat_rec']), '_auc': np.mean(r['pat_auc'])})

perwin_df = pd.DataFrame(win_rows).sort_values('_auc', ascending=False).drop(columns='_auc').reset_index(drop=True)
perpat_df = pd.DataFrame(pat_rows).sort_values('_auc', ascending=False).drop(columns='_auc').reset_index(drop=True)
print('\n[1/2] Tabela PER-OKNO (AUC/Acc = gotowe wyniki sekcji 4/5; średnia ± std po CV):')
print('     (Te same liczby co linear-eval w sekcjach 4/5 - max AUC po epokach, bez ponownego treningu.)')
display(perwin_df)
print('\n[2/2] Tabela PER-PACJENT (głowica z epoki max AUC per-pacjent + early-stopping; średnia ± std po CV):')
print('     (Recall = czułość: jaki odsetek chorych AD wykryto - kluczowe w diagnozie. Metryka rozstrzygająca.)')
display(perpat_df)
perpat_df
